# 01 · Data: download and preprocessing

This notebook says where the datasets of the study come from and under which conditions they can be obtained, downloads the files that are openly accessible,
and turns the recordings into the cached arrays that all later notebooks read.

| Dataset | Version | Used for | Access |
|---|---|---|---|
| PTB-XL (Germany) | 1.0.3 | training with labels, validation, test | open |
| MIMIC-IV-ECG (USA) | 1.0 | unlabelled training pool, test set, pairs of recordings | open |
| MIMIC-IV, patients table | 3.1 | optional: age and sex of the patients in MIMIC-IV-ECG (Table 1) | credentialed users, data use agreement |
| CODE-15% (Brazil) | 1.0.0 | test set, pairs with a change of label | open |
| EchoNext (USA) | 1.1.1 | test set with an echocardiographic reference | registered users, data use agreement |

No data are distributed with this repository. Each section below gives the source, the licence or the conditions of access, the files that are needed, and the folder below `MASTER/data/raw` in which they must lie.
Read the terms of each dataset on its own page before you use it; the summaries here do not replace them.

**Preprocessing.** Every recording is brought to 12 leads and 10 s at 500 Hz (resampled where necessary), filtered as a whole with `filter_ecg` (notebook 00) and stored as float16.
All later steps (segments, windows, scaling of the input of each model) start from these arrays, so the filter is applied exactly once and never to a piece of a recording.

| Section | Download | Raw data on disk | Cached array | Time of preprocessing |
|---|---|---|---|---|
| PTB-XL | by this notebook | 1.8 GB (archive) + 2.6 GB | 2.6 GB | about 3 min (2 min to extract the archive, 1 min to filter) |
| MIMIC-IV-ECG | by this notebook and by notebook 02 | 0.25 GB (tables) + about 5 GB (recordings that notebook 02 examines) or 97 GB (all recordings) | 4.7 GB, written by notebook 02 | in notebook 02 |
| CODE-15% | by this notebook | 45 GB | 40 GB | about 15 min |
| EchoNext | by hand (data use agreement) | 24 GB | 12 GB | about 6 min |

Times are approximate, for 16 worker processes (`N_JOBS` in notebook 00) on the computer described in notebook 00; they do not include the downloads.
Every step skips what already exists, so the notebook can be run again at any time: an interrupted download continues, and an array that was not finished is written again.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import shutil
import zipfile

import h5py
import pandas as pd
import requests
from scipy.signal import resample_poly

## Download of single files

`download_file` is used for all large files. It writes to a temporary name and renames the file at the end, so a file with its final name is always complete;
when a download is interrupted, the next call continues from the bytes that have already arrived.

In [ ]:
def md5_of_file(path):
    """MD5 checksum of a file, read in pieces of 16 MB."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for piece in iter(lambda: f.read(1 << 24), b""):
            h.update(piece)
    return h.hexdigest()


def download_file(url, target, size=None, md5=None):
    """Downloads url to the file `target`. Does nothing when the file exists (with the expected size in bytes, if one is given). Returns True when something was downloaded.
    The data go to `target` + ".part" first. An interrupted download is continued from the end of that file if the server allows it."""
    target = Path(target); part = target.with_name(target.name + ".part")
    if target.exists() and (size is None or target.stat().st_size == size):
        print(f"   {target.name}: present")
        return False
    target.parent.mkdir(parents=True, exist_ok=True); t0 = time.time(); complete = False
    for attempt in range(10):
        have = part.stat().st_size if part.exists() else 0
        if size is not None and have == size:
            complete = True
            break
        headers = {"Accept-Encoding": "identity"}                                   # no compression in transit, so that byte positions and sizes refer to the file itself
        if have:
            headers["Range"] = f"bytes={have}-"
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                if r.status_code == 416:                                            # the server cannot continue from this position: start again from the beginning
                    part.unlink()
                    continue
                if 400 <= r.status_code < 500 and r.status_code not in (408, 429):   # the file is not there or access is refused: trying again does not help
                    raise RuntimeError(f"{target.name}: the server answered with status {r.status_code} for {url}")
                r.raise_for_status()
                resumed = r.status_code == 206                                      # 206: the server sends the rest of the file; 200: it sends the whole file
                expected = int(r.headers["Content-Length"]) + (have if resumed else 0) if "Content-Length" in r.headers else None
                with open(part, "ab" if resumed else "wb") as f:
                    for piece in r.iter_content(chunk_size=1 << 20):
                        f.write(piece)
            wanted = size if size is not None else expected
            if wanted is None or part.stat().st_size == wanted:
                complete = True
                break
        except requests.RequestException as e:
            print(f"   {target.name}: {type(e).__name__}, trying again", flush=True)
        time.sleep(min(60, 5 * (attempt + 1)))
    if not complete:
        raise RuntimeError(f"{target.name}: the download is not complete; run the cell again to continue it")
    if md5 is not None and md5_of_file(part) != md5:
        part.unlink()
        raise RuntimeError(f"{target.name}: wrong MD5 checksum, the file was deleted; run the cell again")
    os.replace(part, target)
    print(f"   {target.name}: downloaded, {target.stat().st_size / 1e6:,.0f} MB in {(time.time() - t0) / 60:.1f} min", flush=True)
    return True

## PTB-XL

**What it is.** 21,799 clinical 12-lead ECGs of 10 s from 18,869 patients, recorded in Germany, with statements assigned by cardiologists and ten recommended folds.
The study uses it for fine-tuning (folds 1–8), for the selection of checkpoints and of decision thresholds (fold 9) and for testing (fold 10).

- Source: PhysioNet, <https://physionet.org/content/ptb-xl/1.0.3/> (version 1.0.3, DOI 10.13026/kfzx-aw45).
- Access: open. Licence: Creative Commons Attribution 4.0 International (the text is in `LICENSE.txt` of the dataset).
- Files needed, in `MASTER/data/raw/ptb-xl/`: `ptbxl_database.csv`, `scp_statements.csv` and the folder `records500/` (the recordings at 500 Hz).

The cell below downloads the archive of the dataset (1.84 GB) and extracts these files only; the recordings at 100 Hz are not needed.
If you already have the dataset, copy or link the three items into `MASTER/data/raw/ptb-xl/` and the cell does nothing.

In [ ]:
PTBXL_ZIP_URL = "https://physionet.org/content/ptb-xl/get-zip/1.0.3/"        # the link "Download the ZIP file" of the dataset page
PTBXL_ZIP = RAW / "ptb-xl-1.0.3.zip"
PTBXL_ZIP_BYTES = 1839504686
PTBXL_NEEDED = ("ptbxl_database.csv", "scp_statements.csv", "LICENSE.txt")      # plus everything below records500/


def ptbxl_complete():
    """True when the two tables and the header and waveform file of every 500-Hz record are in PTBXL_DIR."""
    if not ((PTBXL_DIR / "ptbxl_database.csv").exists() and (PTBXL_DIR / "scp_statements.csv").exists()):
        return False
    names = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", usecols=["filename_hr"]).filename_hr
    return all((PTBXL_DIR / f"{name}.hea").exists() and (PTBXL_DIR / f"{name}.dat").exists() for name in names)


def download_ptbxl():
    if ptbxl_complete():
        print("PTB-XL: present")
        return
    download_file(PTBXL_ZIP_URL, PTBXL_ZIP, size=PTBXL_ZIP_BYTES)
    with zipfile.ZipFile(PTBXL_ZIP) as z:
        for info in z.infolist():
            name = info.filename.split("/", 1)[-1]                           # path below the top folder of the archive
            if info.is_dir() or not (name.startswith("records500/") or name in PTBXL_NEEDED):
                continue
            target = PTBXL_DIR / name
            if target.exists() and target.stat().st_size == info.file_size:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as source, open(target, "wb") as out:
                shutil.copyfileobj(source, out)
    if not ptbxl_complete():
        raise RuntimeError("PTB-XL: files are missing after the extraction of the archive")
    print("PTB-XL: extracted to", PTBXL_DIR, "(the archive", PTBXL_ZIP.name, "is no longer needed and can be deleted)")


download_ptbxl()

### Filtered array of PTB-XL

All 21,799 recordings are cached, in the order of the rows of `ptbxl_database.csv`; which of them are used is decided in notebook 02.
The 500-Hz records are already 10 s long, in millivolts and in the lead order of `LEADS`, so they are only filtered.
Missing samples (none are expected) would be set to zero before filtering, because the filter cannot handle them.

In [ ]:
def _ptbxl_record(path):
    """Worker: one 500-Hz record of PTB-XL -> filtered float16 array (12, 5000)."""
    import wfdb
    signal, _ = wfdb.rdsamp(path)                                            # (5000, 12), mV
    return filter_ecg(np.nan_to_num(signal.T.astype(np.float64))).astype(np.float16)


def build_ptbxl_array():
    if PTBXL_ARRAY.exists():
        print("PTB-XL array: present", np.load(PTBXL_ARRAY, mmap_mode="r").shape)
        return
    names = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", usecols=["filename_hr"]).filename_hr
    write_array(PTBXL_ARRAY, len(names), parallel_map(_ptbxl_record, [str(PTBXL_DIR / name) for name in names]), report_every=5000)
    print("PTB-XL array: written", np.load(PTBXL_ARRAY, mmap_mode="r").shape)


build_ptbxl_array()        # about 1 min with 16 worker processes (about 4 min in one process)

## MIMIC-IV-ECG

**What it is.** 800,035 diagnostic 12-lead ECGs of 10 s at 500 Hz from 161,352 patients of a hospital in the USA, with the interpretation statements of the ECG machine.
The study draws from it the unlabelled pool that is used in distillation, a test set, and two sets of pairs of recordings of the same patient (notebook 02).

- Source: PhysioNet, <https://physionet.org/content/mimic-iv-ecg/1.0/> (version 1.0, DOI 10.13026/4nqg-sb35).
- Access: open. Licence: Open Data Commons Open Database License v1.0 (the text is in `LICENSE.txt` on the dataset page).
- Files needed, in `MASTER/data/raw/mimic-iv-ecg/`: `record_list.csv` (patient, study, time and path of every recording), `machine_measurements.csv` (time and machine statements),
  and the records themselves below `files/`, each with a header file (`.hea`) and a waveform file (`.dat`), for example `files/p1000/p10000032/s40689238/40689238.hea`.

The two tables are downloaded here. For the recordings there are two ways:

1. **Only what is needed (default).** Notebook 02 examines about 40,000 recordings (5 GB) and downloads each of them when it is first needed. Nothing has to be done here.
   At the 5–6 recordings per second that we obtained with 16 connections, this adds about 2 h to notebook 02.
2. **Everything in advance.** Set `DOWNLOAD_ALL_MIMIC_RECORDINGS = True` below to fetch all 800,035 recordings file by file (97 GB on disk; about 40 h at the same rate).
   Notebook 02 then needs no internet connection. The dataset page also offers the whole dataset as one archive (33.8 GB); if you use it, extract it so that `files/` lies in `MASTER/data/raw/mimic-iv-ecg/`.

Both ways lead to the same sets, because the sets depend only on the tables, on the seeds and on the recordings that are examined.
The recordings of MIMIC-IV-ECG are filtered in notebook 02, after the sets have been drawn, because only the selected recordings are cached.

In [ ]:
MIMIC_TABLES = {"record_list.csv": 70403124, "machine_measurements.csv": 182674683}        # file -> size in bytes
for _name, _size in MIMIC_TABLES.items():
    download_file(MIMIC_URL + _name, MIMIC_DIR / _name, size=_size)

In [ ]:
DOWNLOAD_ALL_MIMIC_RECORDINGS = False        # True: download all 800,035 recordings now (97 GB, about 40 h); False: notebook 02 downloads the recordings that it examines

if DOWNLOAD_ALL_MIMIC_RECORDINGS:
    _paths = pd.read_csv(MIMIC_DIR / "record_list.csv", usecols=["path"]).path.tolist()
    print(f"MIMIC-IV-ECG: {download_mimic_records(_paths):,} recordings downloaded, {len(_paths):,} present")       # repeat the cell after an interruption: complete files are skipped
else:
    print("MIMIC-IV-ECG: recordings are downloaded by notebook 02 when they are examined")

### MIMIC-IV, patients table (optional)

MIMIC-IV-ECG contains neither age nor sex. For Table 1 and Supplementary Table 1 they were taken from the patients table of the clinical database MIMIC-IV, which is linked by `subject_id`.

- Source: PhysioNet, <https://physionet.org/content/mimiciv/> (we used version 3.1, DOI 10.13026/kpb9-mt58); file `hosp/patients.csv.gz`.
- Access: credentialed PhysioNet users who have completed the required training and signed the data use agreement of the project (PhysioNet Credentialed Health Data License 1.5.0). See the dataset page for the procedure.
- File needed, in `MASTER/data/raw/mimic-iv/hosp/`: `patients.csv.gz` (columns used: `subject_id`, `gender`, `anchor_age`, `anchor_year`).

This file cannot be downloaded without credentials, so it has to be placed by hand. It is optional: without it everything runs, and age and sex of the MIMIC-IV-ECG sets are reported as not available.

In [ ]:
print("MIMIC-IV patients table:", "present" if MIMIC_PATIENTS.exists() else f"absent (optional; expected at {MIMIC_PATIENTS})")

## CODE-15%

**What it is.** 345,779 12-lead ECG exams from 233,770 patients of a telehealth network in Brazil, a 15% sample of the patients of the CODE dataset, with labels for six abnormalities
(first-degree atrioventricular block, right and left bundle branch block, sinus bradycardia, sinus tachycardia, atrial fibrillation) and a flag for a normal ECG. The dataset page describes how the labels were obtained.
The study uses it for evaluation only.

- Source: Zenodo, <https://zenodo.org/records/4916206> (version 1.0.0, DOI 10.5281/zenodo.4916206).
- Access: open. Licence: Creative Commons Attribution 4.0 International.
- Files needed, in `MASTER/data/raw/code15/`: `exams.csv` and `exams_part0.zip` … `exams_part17.zip` (45 GB). Each archive holds one HDF5 file with the datasets `tracings` (exams × 4,096 samples × 12 leads, 400 Hz)
  and `exam_id`. The archives can stay as they are: every HDF5 file is extracted to a temporary folder when it is processed and deleted afterwards. If you have extracted them yourself, put `exams_part{i}.hdf5` into the same folder instead.

The cell below downloads the files and compares each with the size and the MD5 checksum that Zenodo lists.

In [ ]:
CODE15_RECORD = "https://zenodo.org/api/records/4916206"        # description of the Zenodo record: names, sizes, checksums and links of its files
CODE15_PARTS = list(range(18))


def download_code15(parts=CODE15_PARTS):
    wanted = ["exams.csv"] + [f"exams_part{i}.zip" for i in parts if not (CODE15_DIR / f"exams_part{i}.hdf5").exists()]
    missing = [name for name in wanted if not (CODE15_DIR / name).exists()]
    if not missing:
        print("CODE-15%: present")
        return
    listed = {f["key"]: f for f in requests.get(CODE15_RECORD, timeout=60).json()["files"]}
    for name in missing:                                                      # about 2.7 GB per archive
        download_file(listed[name]["links"]["self"], CODE15_DIR / name, size=listed[name]["size"], md5=listed[name]["checksum"].split(":")[-1])


download_code15()

### Filtered arrays of CODE-15%

A tracing has 4,096 samples at 400 Hz (10.24 s). Exams that are shorter are padded with zeros on both sides.
In the files, 131 thousand exams fill all 4,096 samples, 201 thousand have 2,934 samples (7.3 s, starting at sample 581), about 13 thousand have other lengths, and 670 tracings are empty.
From every exam that is cached, the 4,000 samples in the middle (samples 48–4,047, 10 s) are taken, resampled to 500 Hz (polyphase filter, factor 5/4), which gives 5,000 samples, and filtered.
The amplitudes are used as stored; they are of the order of millivolts (the median peak-to-peak amplitude of lead II is about 2 mV), and every model rescales its input.

**Which exams are cached.** An exam is cached when its tracing has signal, that is, a sample that is not zero in some lead, within samples 100–599.
This leaves out the empty tracings, the exams whose signal starts later than 1.5 s into the tracing, and the last row of every HDF5 file, which is empty and has the exam number 0.
337,071 of the 345,779 exams are cached.

**Length of the signal.** Most cached exams are still shorter than 10 s. The column `signal_samples` of the index gives the number of samples from the first to the last sample that is not zero,
counted within the 4,000 samples of the 10-s window. Notebook 02 uses the complete 10-s exams, whose signal spans at least 3,868 of these samples (`CODE15_MIN_SAMPLES` in notebook 00); there are 132,829.

One array and one index file are written for each of the 18 parts. `part{i}_index.csv` has one line per cached exam: `exam_id`, `row` (row of the array), `signal_samples`, followed by the columns of `exams.csv` for that exam.

In [ ]:
def code15_signal(tracings):
    """tracings: array (n, 4096, 12). Returns two arrays (n,): whether the tracing has signal within samples 100-599, and the number of samples from the first to the last
    non-zero sample within the 10-s window (samples 48-4047; 0 when the window is empty)."""
    present = (tracings != 0).any(axis=2)                                     # (n, 4096): True where some lead is not zero
    window = present[:, 48:4048]
    first = window.argmax(axis=1); last = 3999 - window[:, ::-1].argmax(axis=1)
    return present[:, 100:600].any(axis=1), np.where(window.any(axis=1), last - first + 1, 0)


def _code15_scan(job):
    """Worker: code15_signal for the exams [start, stop) of an HDF5 file."""
    path, start, stop = job
    with h5py.File(path, "r") as h:
        return code15_signal(h["tracings"][start:stop])


def _code15_filter(job):
    """Worker: the exams [start, stop) of an HDF5 file that are kept -> filtered float16 array (k, 12, 5000)."""
    path, start, stop, keep = job
    if not keep.any():
        return np.zeros((0, 12, N10), np.float16)
    with h5py.File(path, "r") as h:
        x = h["tracings"][start:stop][keep]
    x = x[:, 48:4048, :].transpose(0, 2, 1).astype(np.float64)                # the 10-s window, (k, 12, 4000)
    x = resample_poly(x, 5, 4, axis=2)                                        # 400 Hz -> 500 Hz, (k, 12, 5000)
    return np.stack([filter_ecg(recording) for recording in x]).astype(np.float16)


def code15_hdf5(i):
    """Path of exams_part{i}.hdf5 and whether it is a temporary copy: the file in CODE15_DIR if it has been extracted there, otherwise a copy extracted from the archive."""
    extracted = CODE15_DIR / f"exams_part{i}.hdf5"
    if extracted.exists():
        return extracted, False
    folder = CODE15_CACHE / "tmp"; folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(CODE15_DIR / f"exams_part{i}.zip") as z:
        name = next(n for n in z.namelist() if n.endswith(".hdf5"))
        z.extract(name, folder)
    return folder / name, True


def build_code15_part(i, exams, out_dir=None):
    """Writes part{i}_filtered_f16.npy and part{i}_index.csv. `exams` is the table exams.csv."""
    out_dir = CODE15_CACHE if out_dir is None else Path(out_dir)
    array_file, index_file = out_dir / f"part{i}_filtered_f16.npy", out_dir / f"part{i}_index.csv"
    if array_file.exists() and index_file.exists():
        print(f"CODE-15% part {i}: present")
        return
    t0 = time.time(); path, temporary = code15_hdf5(i)
    try:
        with h5py.File(path, "r") as h:
            ids = h["exam_id"][:]
        blocks = [(str(path), start, min(start + 64, len(ids))) for start in range(0, len(ids), 64)]            # the workers read and process 64 exams at a time
        scanned = list(parallel_map(_code15_scan, blocks))
        keep = np.concatenate([s[0] for s in scanned]) & (ids > 0); samples = np.concatenate([s[1] for s in scanned]); n_keep = int(keep.sum())
        if array_file.exists():                                               # the array is complete (see write_array); only the index is missing
            assert np.load(array_file, mmap_mode="r").shape[0] == n_keep, f"{array_file.name} does not fit the exams that are kept; delete it and run again"
        else:
            write_array(array_file, n_keep, parallel_map(_code15_filter, [(p, a, b, keep[a:b]) for p, a, b in blocks]))
        index = pd.DataFrame({"exam_id": ids[keep], "row": np.arange(n_keep), "signal_samples": samples[keep]}).merge(exams, on="exam_id", how="left")
        index.to_csv(index_file, index=False)
    finally:
        if temporary:
            Path(path).unlink(missing_ok=True)
    print(f"CODE-15% part {i}: {len(ids):,} rows in the file, {n_keep:,} exams cached, {int((samples[keep] >= CODE15_MIN_SAMPLES).sum()):,} of them with at least {CODE15_MIN_SAMPLES:,} samples of signal; {(time.time() - t0) / 60:.1f} min", flush=True)


def build_code15_cache(parts=CODE15_PARTS):
    exams = pd.read_csv(CODE15_DIR / "exams.csv")
    for i in parts:
        build_code15_part(i, exams)
    temporary = CODE15_CACHE / "tmp"
    if temporary.exists() and not any(temporary.iterdir()):
        temporary.rmdir()                                                      # the folder for the extracted HDF5 files is empty again


build_code15_cache()        # less than 1 min per part with 16 worker processes, about 15 min for the 18 parts (extraction of the HDF5 files included)

## EchoNext

**What it is.** 100,000 12-lead ECGs from 36,286 patients of a hospital system in the USA, with echocardiographic measurements and labels of structural heart disease that were derived from echocardiograms.
The study uses it for evaluation only, with the flag for a left ventricular wall thickness of 13 mm or more (`lvwt_gte_13_flag`) as a reference that is not read from the ECG.

- Source: PhysioNet, <https://physionet.org/content/echonext/1.1.1/> (version 1.1.1, DOI 10.13026/7cfw-d091).
- Access: restricted. Registered PhysioNet users can download the files after signing the data use agreement of the project on the dataset page
  (PhysioNet Restricted Health Data Use Agreement 1.5.0; licence: PhysioNet Restricted Health Data License 1.5.0, in `LICENSE.txt` of the dataset). See the dataset page for the conditions.
- Files needed, in `MASTER/data/raw/echonext/`: `echonext_metadata_100k.csv` and the four waveform files `EchoNext_train_waveforms.npy`, `EchoNext_val_waveforms.npy`, `EchoNext_test_waveforms.npy`
  and `EchoNext_no_split_waveforms.npy` (24 GB together). The files of tabular features are not used.

These files cannot be downloaded without an account and a signed agreement, so they have to be placed by hand; the dataset page shows the download commands once the agreement is signed.

### Filtered array of EchoNext

A waveform file has the shape (ECGs × 1 × 2,500 samples × 12 leads): 10 s at 250 Hz. The ECGs are resampled to 500 Hz (polyphase filter, factor 2) and filtered.
The provider has already processed the waveforms (median filter for each lead, clipping at the 0.1st and 99.9th percentiles, and scaling with the mean and standard deviation of the whole dataset),
so the amplitudes are not in millivolts. This does not matter here, because every model scales each input with the samples of that input.

The four files are cached one after another (train, val, test, no_split; the splits are those of the provider and have no role in this study).
Within each split, the rows of the waveform file are in the order of the rows of the metadata with that split. `index.csv` has one line per row of the array: `row`, `metadata_row` (row of `echonext_metadata_100k.csv`, counted from 0) and `split`.

In [ ]:
ECHONEXT_SPLITS = ["train", "val", "test", "no_split"]


def _echonext_filter(job):
    """Worker: the ECGs [start, stop) of a waveform file -> filtered float16 array (k, 12, 5000)."""
    path, start, stop = job
    x = np.asarray(np.load(path, mmap_mode="r")[start:stop, 0]).astype(np.float64).transpose(0, 2, 1)    # (k, 12, 2500) at 250 Hz
    x = resample_poly(x, 2, 1, axis=2)                                                                   # 250 Hz -> 500 Hz
    return np.stack([filter_ecg(recording) for recording in x]).astype(np.float16)


def build_echonext_array():
    if ECHONEXT_ARRAY.exists() and ECHONEXT_INDEX.exists():
        print("EchoNext array: present", np.load(ECHONEXT_ARRAY, mmap_mode="r").shape)
        return
    files = [ECHONEXT_DIR / "echonext_metadata_100k.csv"] + [ECHONEXT_DIR / f"EchoNext_{s}_waveforms.npy" for s in ECHONEXT_SPLITS]
    if not all(f.exists() for f in files):
        print("EchoNext: not found; place these files in", ECHONEXT_DIR, "and run the cell again:", ", ".join(f.name for f in files if not f.exists()))
        return
    meta = pd.read_csv(files[0])
    index = pd.concat([pd.DataFrame({"metadata_row": np.where(meta.split.values == s)[0], "split": s}) for s in ECHONEXT_SPLITS], ignore_index=True)
    index.insert(0, "row", np.arange(len(index)))
    jobs = []
    for s, path in zip(ECHONEXT_SPLITS, files[1:]):
        n = np.load(path, mmap_mode="r").shape[0]
        assert n == int((index.split == s).sum()), f"{path.name} has {n} ECGs, the metadata list {int((index.split == s).sum())}"
        jobs += [(str(path), start, min(start + 64, n)) for start in range(0, n, 64)]                           # the workers read and process 64 ECGs at a time
    if not ECHONEXT_ARRAY.exists():
        write_array(ECHONEXT_ARRAY, len(index), parallel_map(_echonext_filter, jobs), report_every=20000)
    index.to_csv(ECHONEXT_INDEX, index=False)
    print("EchoNext array: written", np.load(ECHONEXT_ARRAY, mmap_mode="r").shape)


build_echonext_array()        # about 6 min with 16 worker processes

## What has been written

The cell lists the cached arrays that exist. Notebook 02 needs all of them, and the tables of MIMIC-IV-ECG.

In [ ]:
def describe_cache():
    rows = []
    for name, path in [("PTB-XL", PTBXL_ARRAY), ("EchoNext", ECHONEXT_ARRAY)] + [(f"CODE-15% part {i}", CODE15_CACHE / f"part{i}_filtered_f16.npy") for i in CODE15_PARTS]:
        rows.append({"array": name, "file": str(path.relative_to(MASTER)), "recordings": np.load(path, mmap_mode="r").shape[0] if path.exists() else "missing"})
    return pd.DataFrame(rows)


_cache = describe_cache()
print(_cache.to_string(index=False))
print("CODE-15% exams cached:", f"{sum(r for r in _cache.recordings[2:] if r != 'missing'):,}", "| MIMIC-IV-ECG tables:", all((MIMIC_DIR / n).exists() for n in ("record_list.csv", "machine_measurements.csv")))

## Check of the filter against the ECGFounder repository (optional)

`filter_ecg` (notebook 00) is our own implementation of the preprocessing of the ECGFounder repository. The cell compares it with the original function `filter_bandpass` on 50 recordings of PTB-XL.
It needs the clone of the repository that notebook 03 uses (`MASTER/models/pretrained/ECGFounder`) and is skipped when the clone is not there yet.

In [ ]:
if (ECGFOUNDER_CODE / "util.py").exists() and PTBXL_ARRAY.exists():
    import importlib.util
    import wfdb
    _spec = importlib.util.spec_from_file_location("ecgfounder_util", ECGFOUNDER_CODE / "util.py"); _util = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_util)
    _names = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", usecols=["filename_hr"]).filename_hr
    _rows = np.random.default_rng(0).choice(len(_names), 50, replace=False); _cached = np.load(PTBXL_ARRAY, mmap_mode="r"); _d_original = _d_cache = 0.0
    for _row in _rows:
        _x = np.nan_to_num(wfdb.rdsamp(str(PTBXL_DIR / _names[_row]))[0].T.astype(np.float64))
        _d_original = max(_d_original, float(np.abs(filter_ecg(_x) - _util.filter_bandpass(_x, FS)).max()))
        _d_cache = max(_d_cache, float(np.abs(filter_ecg(_x).astype(np.float16).astype(np.float64) - np.asarray(_cached[_row], dtype=np.float64)).max()))
    print(f"largest absolute difference between filter_ecg and the original function: {_d_original:.3g} mV; between filter_ecg and the cached array: {_d_cache:.3g} mV")
else:
    print("skipped: the clone of the ECGFounder repository (notebook 03) or the PTB-XL array is not present")

## File formats

**Cached arrays** (`.npy`, float16, shape `(n, 12, 5000)`): `n` recordings × 12 leads in the order I, II, III, aVR, aVL, aVF, V1–V6 × 10 s at 500 Hz, filtered. Units are millivolts, except for EchoNext (see above).
The arrays are opened with `np.load(path, mmap_mode="r")`, which reads only the rows that are used.

| File (below `MASTER/data/derived/`) | Rows |
|---|---|
| `ptbxl_filtered_f16.npy` | 21,799, in the order of `ptbxl_database.csv` |
| `code15/part{i}_filtered_f16.npy`, i = 0 … 17 | 337,071 in total; the exams of `exams_part{i}.hdf5` that are cached, in the order of that file |
| `echonext/filtered_f16.npy` | 100,000, in the order of `echonext/index.csv` |
| `mimic_pool_f16.npy`, `mimic_eval_f16.npy` | written by notebook 02 |

**`code15/part{i}_index.csv`**: one line per row of the array of that part.

| Column | Content |
|---|---|
| `exam_id` | number of the exam (as in `exams.csv`) |
| `row` | row of `part{i}_filtered_f16.npy` |
| `signal_samples` | samples from the first to the last non-zero sample within the 10-s window (at 400 Hz; 4,000 for a complete exam) |
| `age`, `is_male`, `nn_predicted_age`, `1dAVb`, `RBBB`, `LBBB`, `SB`, `ST`, `AF`, `patient_id`, `death`, `timey`, `normal_ecg`, `trace_file` | the columns of `exams.csv` for this exam |

**`echonext/index.csv`**: one line per row of the array.

| Column | Content |
|---|---|
| `row` | row of `filtered_f16.npy` |
| `metadata_row` | row of `echonext_metadata_100k.csv` (counted from 0) that describes this ECG |
| `split` | split of the provider (train, val, test, no_split) |